In [1]:
# Cell 2: Imports (without aliases) and plot styling configuration
import pandas
import numpy
import gseapy
import mygene
import matplotlib.pyplot

# Global matplotlib settings to enforce a crisp white background for presentation
matplotlib.pyplot.rcParams['figure.facecolor'] = 'white'
matplotlib.pyplot.rcParams['axes.facecolor'] = 'white'
matplotlib.pyplot.rcParams['savefig.facecolor'] = 'white'
matplotlib.pyplot.style.use('default')

In [2]:
# Cell 4: Load DESeq2 output and generate the ranked list
input_file = "/Users/adrian/research/bmcbf/073_bologna/results/degs/effect_timepoint_D7_vs_D0.full.tsv"

# Cell 3: Step 1 - Load DESeq2 output and extract raw gene IDs
# Load DESeq2 CSV file (assumes gene IDs are in index; adjust index_col or column name if needed)
deseq_df = pandas.read_csv(input_file, index_col=0, sep='\t')
deseq_df.head()

,gene_id,baseMean,log2FC_MLE,lfcSE,stat,pvalue,padj,log2FC_shr,delta_counts,median_TPM_A,median_TPM_B,ensembl_id,gene_name,biotype,description2
ENSG00000000003,ENSG00000000003.15,2564.415317,-0.261314,0.154118,2.903878,8.836672e-02,1.959405e-01,-0.246384,-475.392692,34.817394,31.055016,ENSG00000000003,TSPAN6,protein_coding,tetraspanin 6
ENSG00000000419,ENSG00000000419.14,1417.302201,-0.213999,0.130845,2.695943,1.006040e-01,2.162919e-01,-0.204786,-211.493064,64.324292,59.887071,ENSG00000000419,DPM1,protein_coding,dolichyl-phosphate mannosyltransferase subunit...
ENSG00000000457,ENSG00000000457.14,445.718268,0.258923,0.216741,1.411169,2.348625e-01,4.013692e-01,0.232241,67.752777,4.081097,5.123647,ENSG00000000457,SCYL3,protein_coding,SCY1 like pseudokinase 3
ENSG00000000460,ENSG00000000460.17,160.754863,0.055311,0.457291,0.014866,9.029582e-01,9.470969e-01,0.036671,32.207333,2.588682,3.360438,ENSG00000000460,C1orf112,protein_coding,chromosome 1 open reading frame 112
ENSG00000000971,ENSG00000000971.17,1511.704485,1.441016,0.152526,81.998958,1.361584e-19,3.323725e-18,1.417419,1368.723365,10.628233,31.584042,ENSG00000000971,CFH,protein_coding,complement factor H


In [11]:
deseq_df['stat'].max()

np.float64(2616.27467380923)

In [14]:
# Cell 4: Step 2 - Calculate ranking metric
if 'stat' in deseq_df.columns:
    #deseq_df['rank_metric'] = deseq_df['stat']
    deseq_df['rank_metric'] = deseq_df['stat'] * numpy.sign(deseq_df['log2FC_shr'])
    print('stat found')
elif 'log2FC_shr' in deseq_df.columns and 'pvalue' in deseq_df.columns:
    pvals = deseq_df['pvalue'].replace(0, 1e-300)
    deseq_df['rank_metric'] = numpy.sign(deseq_df['log2FC_shr']) * (-numpy.log10(pvals))
    print('found both')
elif 'log2FC_shr' in deseq_df.columns:
    deseq_df['rank_metric'] = deseq_df['log2FC_shr']
else:
    raise KeyError("Could not find 'stat', 'log2FoldChange', or 'pvalue' columns in DESeq2 table.")

# Clean invalid or missing values
deseq_df = deseq_df.replace([numpy.inf, -numpy.inf], numpy.nan)
deseq_df = deseq_df.dropna(subset=['gene_name', 'rank_metric'])
deseq_df.head()

stat found


,gene_id,baseMean,log2FC_MLE,lfcSE,stat,pvalue,padj,log2FC_shr,delta_counts,median_TPM_A,median_TPM_B,ensembl_id,gene_name,biotype,description2,rank_metric,abs_metric
ENSG00000000003,ENSG00000000003.15,2564.415317,-0.261314,0.154118,2.903878,8.836672e-02,1.959405e-01,-0.246384,-475.392692,34.817394,31.055016,ENSG00000000003,TSPAN6,protein_coding,tetraspanin 6,-2.903878,2.903878
ENSG00000000419,ENSG00000000419.14,1417.302201,-0.213999,0.130845,2.695943,1.006040e-01,2.162919e-01,-0.204786,-211.493064,64.324292,59.887071,ENSG00000000419,DPM1,protein_coding,dolichyl-phosphate mannosyltransferase subunit...,-2.695943,2.695943
ENSG00000000457,ENSG00000000457.14,445.718268,0.258923,0.216741,1.411169,2.348625e-01,4.013692e-01,0.232241,67.752777,4.081097,5.123647,ENSG00000000457,SCYL3,protein_coding,SCY1 like pseudokinase 3,1.411169,1.411169
ENSG00000000460,ENSG00000000460.17,160.754863,0.055311,0.457291,0.014866,9.029582e-01,9.470969e-01,0.036671,32.207333,2.588682,3.360438,ENSG00000000460,C1orf112,protein_coding,chromosome 1 open reading frame 112,0.014866,0.014866
ENSG00000000971,ENSG00000000971.17,1511.704485,1.441016,0.152526,81.998958,1.361584e-19,3.323725e-18,1.417419,1368.723365,10.628233,31.584042,ENSG00000000971,CFH,protein_coding,complement factor H,81.998958,81.998958


In [17]:
# Cell 5: Deduplicate gene symbols and create the ranked list
# If duplicate gene_names exist, keep the row with the largest absolute ranking metric
deseq_df['abs_metric'] = deseq_df['rank_metric'].abs()
deseq_dedup = deseq_df.sort_values('abs_metric', ascending=False).drop_duplicates('gene_name')

# Generate final sorted pandas Series (gene_name as index, ranking metric as value)
rnk = deseq_dedup.set_index('gene_name')['rank_metric'].sort_values(ascending=False)

print(f"Total unique ranked genes ready for GSEA: {len(rnk)}")
print(max(rnk), min(rnk))

Total unique ranked genes ready for GSEA: 14862
2616.27467380923 -1592.6003863286


In [18]:
# Cell 6: Run Preranked GSEA
gene_sets = ['MSigDB_Hallmark_2020', 'KEGG_2021_Human', 'Reactome_2022']

pre_res = gseapy.prerank(
    rnk=rnk,
    gene_sets=gene_sets,
    organism='human',
    permutation_num=1000,
    seed=42,
    outdir=None
)

# Display top enriched terms table
pre_res.res2d.head(10)

2026-09-16 09:53:01,966 [WARNING] Duplicated values found in preranked stats: 0.01% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,MSigDB_Hallmark_2020__TNF-alpha Signaling via ...,-0.761941,-1.685575,0.000000,0.004190,0.003,52/183,4.62%,PHLDA1;PLAU;PLAUR;IER3;INHBA;EGR1;GFPT2;LIF;VE...
1,prerank,MSigDB_Hallmark_2020__IL-2/STAT5 Signaling,-0.753505,-1.628333,0.002242,0.015365,0.022,33/159,4.31%,PHLDA1;NT5E;ADAM19;NRP1;LIF;CD44;TGM2;SPP1;ENP...
2,prerank,Reactome_2022__Degradation Of GLI1 By Proteaso...,-0.845007,-1.603838,0.011696,1.000000,0.644,30/56,10.01%,PSMD2;NUMB;PSMB7;PSMA5;PSMC1;PRKCA;GLI1;UBB;PS...
3,prerank,Reactome_2022__Mitotic G2-G2/M Phases R-HSA-45...,-0.725877,-1.602031,0.004367,1.000000,0.655,37/180,10.01%,PHLDA1;PSMD2;CSNK1E;HSP90AA1;PSMB7;PSMA5;PSMC1...
4,prerank,Reactome_2022__Role Of GTSE1 In G2/M Progressi...,-0.843529,-1.599005,0.005952,1.000000,0.681,28/57,10.01%,PSMD2;HSP90AA1;PSMB7;PSMA5;PSMC1;UBB;HSP90AB1;...
5,prerank,Reactome_2022__G2/M Transition R-HSA-69275,-0.725363,-1.597961,0.004357,0.808189,0.687,36/178,10.01%,PHLDA1;PSMD2;CSNK1E;HSP90AA1;PSMB7;PSMA5;PSMC1...
6,prerank,Reactome_2022__ER-Phagosome Pathway R-HSA-1236974,-0.812374,-1.587999,0.008621,0.801822,0.751,33/78,10.01%,PSMD2;PDIA3;CALR;LY96;PSMB7;PSMA5;PSMC1;UBB;PS...
7,prerank,Reactome_2022__Dectin-1 Mediated Noncanonical ...,-0.833183,-1.579901,0.018330,0.803739,0.805,29/57,10.01%,PSMD2;PSMB7;PSMA5;PSMC1;UBB;PSMD11;PSMD3;PSMD1...
8,prerank,Reactome_2022__Antigen processing-Cross Presen...,-0.791403,-1.576495,0.010571,0.738994,0.826,35/87,10.01%,CTSL;PSMD2;PDIA3;CALR;LY96;PSMB7;PSMA5;PSMC1;U...
9,prerank,Reactome_2022__Signal Transduction By L1 R-HSA...,-0.941856,-1.575753,0.005906,0.657060,0.830,4/18,2.79%,ITGB3;NRP1;ITGA5;ITGB1


In [27]:
pre_res.res2d[pre_res.res2d['FDR q-val'] < 0.05]

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,MSigDB_Hallmark_2020__TNF-alpha Signaling via ...,-0.761941,-1.685575,0.000000,0.004190,0.003,52/183,4.62%,PHLDA1;PLAU;PLAUR;IER3;INHBA;EGR1;GFPT2;LIF;VE...
1,prerank,MSigDB_Hallmark_2020__IL-2/STAT5 Signaling,-0.753505,-1.628333,0.002242,0.015365,0.022,33/159,4.31%,PHLDA1;NT5E;ADAM19;NRP1;LIF;CD44;TGM2;SPP1;ENP...


In [ ]:

# Cell 1: Recalculate metric and set FDR threshold value
passing_res['log10_1_over_fdr'] = numpy.log10(1 / passing_res['FDR q-val'].replace(0, 1e-300))

# Significance threshold at FDR = 0.25
fdr_threshold_val = numpy.log10(1 / 0.25)  # 0.602

passing_res[['Term', 'NES', 'FDR q-val', 'log10_1_over_fdr']].head()

In [ ]:
passing_res['NES'].max

In [ ]:
# Cell 2: Fixed gseapy.dotplot (cutoff=100 overrides gseapy internal filtering)
num_terms = len(passing_res)

if num_terms > 0:
    fig_height = max(6, num_terms * 0.35)
    fig, ax = matplotlib.pyplot.subplots(figsize=(8, fig_height), facecolor='white')
    ax.set_facecolor('white')

    # Setting cutoff=100 ensures gseapy does not filter out log10_1_over_fdr values (> 0.602)
    gseapy.dotplot(
        passing_res,
        column="log10_1_over_fdr",
        title="Enriched Pathways (Color: -log10 FDR)",
        top_term=num_terms,
        cutoff=100,
        cmap="viridis",
        ax=ax
    )

    matplotlib.pyplot.tight_layout()
    matplotlib.pyplot.savefig("passing_pathways_dotplot_viridis.png", dpi=300, bbox_inches='tight', facecolor='white')
    matplotlib.pyplot.show()